# Tuần 2 · Huấn luyện và validation

So sánh năm mô hình phân loại trên tập train đã xác định ở [tuần 1](01_eda.ipynb). Dùng cùng 5 fold có phân tầng và xếp hạng theo Average Precision (AP) của lớp mua `Revenue=True`. Tập test dành cho đánh giá cuối ở tuần 3.

Chạy từ thư mục gốc của project với kernel Python trong `.venv`.

## 1. Đọc và xác minh split

SHA-256 xác nhận CSV vẫn có đúng nội dung và thứ tự dòng khi split được tạo. Danh sách vị trí được kiểm tra để xác nhận mỗi dòng thuộc đúng một tập. Chỉ lấy dòng train để tạo feature và nhãn.

In [1]:
from hashlib import sha256
import json
from pathlib import Path

from IPython.display import display
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

csv_path = Path("data/online_shoppers_intention.csv")
split_path = Path("data/split_indices.json")
split = json.loads(split_path.read_text(encoding="utf-8"))
if sha256(csv_path.read_bytes()).hexdigest() != split["csv_sha256"]:
    raise ValueError("CSV đã thay đổi so với split của tuần 1.")

df = pd.read_csv(csv_path)
train_indices = split["train_indices"]
test_indices = split["test_indices"]
all_indices = train_indices + test_indices
if (
    not train_indices
    or not test_indices
    or any(type(index) is not int for index in all_indices)
    or len(all_indices) != len(df)
    or split["n_rows"] != len(df)
    or set(all_indices) != set(df.index)
):
    raise ValueError("Split phải chứa đủ vị trí dòng và mỗi vị trí chỉ xuất hiện một lần.")

train = df.iloc[train_indices]
X_train = train.drop(columns=["Revenue", "PageValues"])
y_train = train["Revenue"]

print(f"scikit-learn {sklearn.__version__}")
print(f"Train có {len(train):,} phiên, {X_train.shape[1]} feature và {y_train.mean():.2%} phiên mua.")
print(f"Split giữ lại {len(test_indices):,} vị trí cho tập test.")

scikit-learn 1.9.1
Train có 9,864 phiên, 16 feature và 15.47% phiên mua.
Split giữ lại 2,466 vị trí cho tập test.


## 2. Preprocessing và fold validation

One-hot encode các biến phân loại, kể cả cột mã số như `Browser` và `TrafficType`. Dùng ma trận dense vì số cột sau khi encode còn nhỏ và cả năm mô hình đều dùng được. `StandardScaler` chỉ áp dụng cho nhóm biến số của Logistic Regression và kNN. Các mô hình cây dùng thang số gốc.

Mỗi lần fit, `cross_validate` tạo một bản sao của cả pipeline và fit encoder, scaler (nếu có) và mô hình trên phần training của fold. Mọi mô hình dùng đúng các cặp vị trí fold đã tạo bên dưới. AP tóm tắt precision theo các mức recall khi xếp hạng phiên bằng xác suất mua, phù hợp khi lớp mua chiếm tỷ lệ thấp.

In [2]:
categorical_columns = [
    "Month", "OperatingSystems", "Browser", "Region",
    "TrafficType", "VisitorType", "Weekend",
]
numeric_columns = [col for col in X_train.columns if col not in categorical_columns]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(X_train, y_train))

fold_summary = []
for fold_number, (fit_indices, val_indices) in enumerate(folds, start=1):
    validation_labels = y_train.iloc[val_indices]
    fold_summary.append({
        "Fold": fold_number,
        "Số phiên fit": len(fit_indices),
        "Số phiên validation": len(val_indices),
        "Số phiên mua validation": int(validation_labels.sum()),
    })
display(pd.DataFrame(fold_summary))

def evaluate_model(name, classifier, scale_numeric=False):
    preprocessing = ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_columns),
        ("numeric", StandardScaler() if scale_numeric else "passthrough", numeric_columns),
    ])
    pipeline = Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", classifier),
    ])
    scores = cross_validate(
        pipeline, X_train, y_train, cv=folds, scoring="average_precision",
        n_jobs=1, error_score="raise",
    )
    ap_scores = scores["test_score"]
    return {
        "model": name,
        "scale_numeric": scale_numeric,
        "parameters": json.dumps(classifier.get_params(), sort_keys=True),
        "mean_average_precision": ap_scores.mean(),
        "std_average_precision": ap_scores.std(ddof=0),
        "mean_fit_seconds": scores["fit_time"].mean(),
        "mean_score_seconds": scores["score_time"].mean(),
        **{f"ap_fold_{i}": score for i, score in enumerate(ap_scores, start=1)},
    }

,Fold,Số phiên fit,Số phiên validation,Số phiên mua validation
0,1,7891,1973,305
1,2,7891,1973,305
2,3,7891,1973,305
3,4,7891,1973,306
4,5,7892,1972,305


## 3. Baseline và Logistic Regression

`DummyClassifier(strategy="prior")` trả cùng xác suất mua cho mọi phiên trong mỗi fold. Điểm AP của baseline vì vậy bằng tỷ lệ phiên mua trong fold validation. Logistic Regression là mô hình đầu tiên học từ feature.

In [3]:
comparison_columns = [
    "model", "mean_average_precision", "std_average_precision",
    "mean_fit_seconds", "mean_score_seconds",
]
baseline_results = [
    evaluate_model("DummyClassifier", DummyClassifier(strategy="prior")),
    evaluate_model(
        "Logistic Regression", LogisticRegression(max_iter=1000), scale_numeric=True
    ),
]
display(pd.DataFrame(baseline_results)[comparison_columns].round(4))

,model,mean_average_precision,std_average_precision,mean_fit_seconds,mean_score_seconds
0,DummyClassifier,0.1547,0.0002,0.0098,0.005
1,Logistic Regression,0.3232,0.0196,0.0556,0.006


## 4. kNN, Decision Tree và Random Forest

Dùng cấu hình ban đầu để so sánh trên CPU. kNN dùng 25 láng giềng. Giới hạn độ sâu và số mẫu tối thiểu ở nút lá cho hai mô hình cây. Random Forest gồm 100 cây.

In [4]:
additional_results = [
    evaluate_model("kNN", KNeighborsClassifier(n_neighbors=25), scale_numeric=True),
    evaluate_model(
        "Decision Tree", DecisionTreeClassifier(
            max_depth=5, min_samples_leaf=20, random_state=42
        )
    ),
    evaluate_model(
        "Random Forest", RandomForestClassifier(
            n_estimators=100, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=1
        )
    ),
]

results = (
    pd.DataFrame(baseline_results + additional_results)
    .sort_values("mean_average_precision", ascending=False, kind="stable")
    .reset_index(drop=True)
)
display(results[comparison_columns].round(4))

report_dir = Path("reports")
report_dir.mkdir(exist_ok=True)
scores_path = report_dir / "validation_scores.csv"
results.to_csv(scores_path, index=False, encoding="utf-8", float_format="%.8f")
print(f"Đã lưu điểm từng fold, tham số và thời gian vào {scores_path.as_posix()}.")

,model,mean_average_precision,std_average_precision,mean_fit_seconds,mean_score_seconds
0,Random Forest,0.3762,0.0294,0.3883,0.0206
1,Logistic Regression,0.3232,0.0196,0.0556,0.0060
2,Decision Tree,0.3180,0.0155,0.0336,0.0063
3,kNN,0.2839,0.0217,0.0138,0.0381
4,DummyClassifier,0.1547,0.0002,0.0098,0.0050


Đã lưu điểm từng fold, tham số và thời gian vào reports/validation_scores.csv.


Độ lệch chuẩn AP được tính trên 5 điểm fold với `ddof=0`. Thời gian là số giây trung bình mỗi fold, gồm thời gian fit pipeline và thời gian tính AP ở hai cột riêng. Giá trị có thể thay đổi theo máy và tải CPU.

## 5. Kết quả sơ bộ

Thứ hạng chỉ dựa vào AP trên validation. Các điểm fold cho biết kết quả dao động thế nào giữa các phần của tập train.

In [5]:
top = results.iloc[0]
baseline = results.loc[results["model"].eq("DummyClassifier")].iloc[0]
print(f"AP cao nhất trên validation là {top['mean_average_precision']:.4f} ({top['model']}).")
print(f"AP của baseline là {baseline['mean_average_precision']:.4f}.")

AP cao nhất trên validation là 0.3762 (Random Forest).
AP của baseline là 0.1547.


Xem tóm tắt tiến độ, giới hạn và công việc tuần 3 trong [Milestone](reports/milestone.md).